In [51]:
import pandas as pd
import numpy as np
import random
import time
import os
from datetime import datetime
import multiprocessing as mp

def apply_nonlinear_transformations(df, score, complexity_level):
    """
    Apply progressive non-linear transformations based on complexity level (0-50).
    Each level adds additional complexity to the pricing model.
    All transformations maintain positive house prices.
    """
    if complexity_level == 0:
        return score

    enhanced_score = score.copy()

    # Level 1-5: Basic non-linear transformations
    if complexity_level >= 1:
        enhanced_score += 30 * np.sqrt(df['TotalArea'])

    if complexity_level >= 2:
        enhanced_score += 3000 * np.log1p(df['LotSize'] / 1000)

    if complexity_level >= 3:
        age_bonus = 15000 * np.exp(-df['HouseAge'] / 25)
        enhanced_score += age_bonus

    if complexity_level >= 4:
        optimal_bedrooms = 3.5
        bedroom_penalty = np.maximum(0, 2000 * (4 - np.abs(df['Bedrooms'] - optimal_bedrooms)))
        enhanced_score += bedroom_penalty

    if complexity_level >= 5:
        school_bonus = 10000 * (1 / (1 + np.exp(-(df['SchoolRating'] - 6))))
        enhanced_score += school_bonus

    # Level 6-10: Interaction effects
    if complexity_level >= 6:
        pool_value = df['HasPool'] * np.sqrt(df['TotalArea']) * 15
        enhanced_score += pool_value

    if complexity_level >= 7:
        fireplace_bonus = df['HasFireplace'] * np.sqrt(df['TotalArea']) * 10
        enhanced_score += fireplace_bonus

    if complexity_level >= 8:
        crime_penalty = np.minimum(15000, 1000 * np.exp(df['CrimeRate'] / 3))
        enhanced_score -= crime_penalty

    if complexity_level >= 9:
        zip_multipliers = {
            '90210': 0.3, '10001': 1.5, '60614': 1.0,
            '94105': 1.2, '77005': 0.8, '30303': 0.6, '98101': 1.1
        }
        distance_penalty = df.apply(
            lambda row: zip_multipliers.get(row['ZipCode'], 1.0) *
                        np.minimum(200, row['DistanceToDowntown']) * 100,
            axis=1
        )
        enhanced_score -= distance_penalty

    if complexity_level >= 10:
        garage_bonus = np.where(
            df['GarageSize'] > 400,
            np.sqrt(df['GarageSize']) * 200,
            np.sqrt(df['GarageSize']) * 100
        )
        enhanced_score += garage_bonus

    # Level 11-15: Complex interactions
    if complexity_level >= 11:
        interaction_bonus = (df['Bedrooms'] * df['Bathrooms'] * np.sqrt(df['TotalArea'])) * 20
        enhanced_score += interaction_bonus

    if complexity_level >= 12:
        seasonal_multipliers = {'Spring': 1.03, 'Summer': 1.05, 'Fall': 1.01, 'Winter': 0.98}
        zip_seasonal_bonus = {
            '90210': 1.1, '10001': 1.0, '60614': 0.95, '94105': 1.05,
            '77005': 1.0, '30303': 0.97, '98101': 1.02
        }
        seasonal_effect = df.apply(
            lambda row: seasonal_multipliers.get(row['SaleSeason'], 1.0) *
                        zip_seasonal_bonus.get(row['ZipCode'], 1.0) * 3000,
            axis=1
        )
        enhanced_score += seasonal_effect

    if complexity_level >= 13:
        current_year = 2024
        renovation_effect = np.where(
            df['RenovatedYear'] == 0, 0,
            np.where(df['RenovatedYear'] > 2010,
                     (df['RenovatedYear'] - 2000) * 500,
                     np.where(df['RenovatedYear'] < 1970,
                              np.maximum(0, (1970 - df['RenovatedYear']) * 100),
                              0))
        )
        enhanced_score += renovation_effect

    if complexity_level >= 14:
        material_multipliers = {'Wood': 1.05, 'Tile': 1.03, 'Carpet': 0.98, 'Mixed': 1.01}
        material_bonus = []
        for idx, row in df.iterrows():
            material_mult = material_multipliers.get(row['FloorMaterial'], 1.0)
            material_bonus.append(material_mult * 2000)
        enhanced_score += pd.Series(material_bonus, index=enhanced_score.index)

    if complexity_level >= 15:
        optimal_duration = 45
        duration_bonus = np.maximum(0, 2000 - 20 * np.abs(df['ListingDuration'] - optimal_duration))
        enhanced_score += duration_bonus

    # Level 16-20: Environmental and location complexities
    if complexity_level >= 16:
        park_bonus = 800 * (1.5 ** df['NearbyParks'] - 1)
        enhanced_score += park_bonus

    if complexity_level >= 17:
        hoa_effect = np.where(
            df['MonthlyHOAFee'] > 500,
            -df['MonthlyHOAFee'] * 2,
            np.where(df['MonthlyHOAFee'] > 100,
                     df['MonthlyHOAFee'] * 3,
                     0)
        )
        enhanced_score += hoa_effect

    if complexity_level >= 18:
        optimal_pool_size = 400
        pool_size_bonus = np.where(
            df['HasPool'] == 1,
            np.maximum(0, 3000 - 5 * np.abs(df['PoolSize'] - optimal_pool_size)),
            0
        )
        enhanced_score += pool_size_bonus

    if complexity_level >= 19:
        quality_values = {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3}
        quality_numeric = df['ExteriorQuality'].map(quality_values)
        quality_bonus = 3000 * (1.5 ** quality_numeric)
        enhanced_score += quality_bonus

    if complexity_level >= 20:
        fireplace_effect = np.where(
            df['FireplaceCount'] > 0,
            5000 * np.log1p(df['FireplaceCount'] * 2),
            0
        )
        enhanced_score += fireplace_effect

    # Level 25
    if complexity_level >= 25:
        zip_premiums = {
            '90210': 80000, '10001': 60000, '94105': 55000,
            '60614': 15000, '98101': 10000, '77005': 8000, '30303': 0
        }
        zip_bonus = df['ZipCode'].map(zip_premiums).fillna(0)
        enhanced_score += zip_bonus

    # Level 30
    if complexity_level >= 30:
        efficiency_bonus = np.where(
            df['TotalArea'] < 4000,
            30 * np.sqrt(df['TotalArea']),
            30 * np.sqrt(4000)
        )
        enhanced_score += efficiency_bonus

    # Level 40
    if complexity_level >= 40:
        inefficiency_adjustment = np.sin(df.index * 0.1) * 2000
        enhanced_score += inefficiency_adjustment

    # Level 50
    if complexity_level >= 50:
        if enhanced_score.std() > 0:
            normalized_score = (enhanced_score - enhanced_score.mean()) / enhanced_score.std()
            recursive_bonus = np.tanh(normalized_score) * enhanced_score * 0.05
            enhanced_score += recursive_bonus

    return enhanced_score

def generate_chunk_data_to_file(args):
    chunk_index, chunk_size, seed, price_noise_multiplier, feature_noise_multiplier, data_type, num_classes, output_path, verbose, complexity_level = args

    np.random.seed(seed + chunk_index)
    random.seed(seed + chunk_index)

    def generate_zipcode():
        return random.choice(['90210', '10001', '60614', '94105', '77005', '30303', '98101'])

    def generate_floor_material():
        return random.choice(['Wood', 'Tile', 'Carpet', 'Mixed'])

    def generate_exterior_quality():
        return random.choice(['Poor', 'Fair', 'Good', 'Excellent'])

    def generate_sale_season():
        return random.choice(['Spring', 'Summer', 'Fall', 'Winter'])

    data = {
        'Bedrooms': np.random.randint(1, 6, chunk_size),
        'Bathrooms': np.random.randint(1, 5, chunk_size),
        'TotalArea': np.random.normal(2000, 500, chunk_size).clip(500, 5000),
        'GarageSize': np.random.normal(300, 100, chunk_size).clip(0, 1000),
        'HasPool': np.random.choice([0, 1], chunk_size, p=[0.8, 0.2]),
        'PoolSize': np.random.normal(400, 150, chunk_size).clip(0, 1000),
        'LotSize': np.random.normal(8000, 3000, chunk_size).clip(2000, 20000),
        'HouseAge': np.random.randint(0, 100, chunk_size),
        'RenovatedYear': np.random.choice(list(range(1950, 2025)) + [0], chunk_size),
        'HasFireplace': np.random.choice([0, 1], chunk_size, p=[0.6, 0.4]),
        'FireplaceCount': np.random.randint(0, 4, chunk_size),
        'ZipCode': [generate_zipcode() for _ in range(chunk_size)],
        'SchoolRating': np.random.randint(1, 11, chunk_size),
        'CrimeRate': np.random.normal(3, 1.5, chunk_size).clip(0, 10),
        'NearbyParks': np.random.randint(0, 6, chunk_size),
        'DistanceToDowntown': np.random.normal(10, 5, chunk_size).clip(0, 50),
        'MonthlyHOAFee': np.random.normal(150, 50, chunk_size).clip(0, 1000),
        'ExteriorQuality': [generate_exterior_quality() for _ in range(chunk_size)],
        'FloorMaterial': [generate_floor_material() for _ in range(chunk_size)],
        'SaleSeason': [generate_sale_season() for _ in range(chunk_size)],
        'ListingDuration': np.random.randint(5, 180, chunk_size)
    }

    df = pd.DataFrame(data)

    # Apply feature noise if specified
    if feature_noise_multiplier > 0:
        numerical_noise_std = {
            'TotalArea': 50,
            'GarageSize': 20,
            'PoolSize': 30,
            'LotSize': 100,
            'HouseAge': 3,
            'CrimeRate': 0.5,
            'MonthlyHOAFee': 10,
            'DistanceToDowntown': 1,
        }

        for col, std in numerical_noise_std.items():
            df[col] += np.random.normal(0, std * feature_noise_multiplier, chunk_size)
            # Ensure values stay within reasonable bounds
            if col == 'TotalArea':
                df[col] = df[col].clip(500, 5000)
            elif col == 'GarageSize':
                df[col] = df[col].clip(0, 1000)
            elif col == 'PoolSize':
                df[col] = df[col].clip(0, 1000)
            elif col == 'LotSize':
                df[col] = df[col].clip(2000, 20000)
            elif col == 'HouseAge':
                df[col] = df[col].clip(0, 100)
            elif col == 'CrimeRate':
                df[col] = df[col].clip(0, 10)
            elif col == 'MonthlyHOAFee':
                df[col] = df[col].clip(0, 1000)
            elif col == 'DistanceToDowntown':
                df[col] = df[col].clip(0, 50)

        # Add noise to binary features
        for col in ['HasPool', 'HasFireplace']:
            flip_mask = np.random.rand(chunk_size) < feature_noise_multiplier * 0.1  # Reduced noise for binary
            df.loc[flip_mask, col] = 1 - df.loc[flip_mask, col]

        # Add noise to categorical features
        for col, generator in [('ZipCode', generate_zipcode),
                               ('ExteriorQuality', generate_exterior_quality),
                               ('FloorMaterial', generate_floor_material),
                               ('SaleSeason', generate_sale_season)]:
            mask = np.random.rand(chunk_size) < feature_noise_multiplier * 0.1  # Reduced noise for categorical
            df.loc[mask, col] = [generator() for _ in range(mask.sum())]

    # Calculate base linear score with positive baseline
    base_price = 150000  # Minimum reasonable house price
    
    score = (
        base_price +
        50000 * df['Bedrooms'] +
        40000 * df['Bathrooms'] +
        100 * df['TotalArea'] +
        50 * df['GarageSize'] +
        10000 * df['HasPool'] +
        20 * df['PoolSize'] +
        0.5 * df['LotSize'] +
        np.maximum(0, 1000 * (50 - df['HouseAge'])) +  # Age penalty, but not too harsh
        3000 * df['HasFireplace'] +
        2000 * df['FireplaceCount'] +
        5000 * df['SchoolRating'] +
        np.maximum(0, 3000 * (5 - df['CrimeRate'])) +  # Crime bonus for low crime
        1000 * df['NearbyParks'] +
        np.maximum(0, 800 * (30 - df['DistanceToDowntown'])) +  # Distance bonus for close locations
        np.maximum(0, 200 * (200 - df['MonthlyHOAFee'])) +  # HOA fee consideration
        np.maximum(0, 100 * (90 - df['ListingDuration']))  # Quick sale bonus
    )

    # Apply non-linear transformations based on complexity level
    enhanced_score = apply_nonlinear_transformations(df, score, complexity_level)
    
    # Ensure all scores are positive (add safety margin)
    min_price = 100000  # Minimum reasonable house price
    enhanced_score = np.maximum(enhanced_score, min_price)

    # Add price noise while maintaining positivity
    if price_noise_multiplier > 0:
        # Use multiplicative noise to maintain positive values
        noise_factor = 1 + np.random.normal(0, price_noise_multiplier, chunk_size)
        noise_factor = np.maximum(noise_factor, 0.5)  # Prevent extreme negative multipliers
        noisy_score = enhanced_score * noise_factor
    else:
        noisy_score = enhanced_score
    
    # Final safety check - ensure all prices are reasonable
    noisy_score = np.maximum(noisy_score, min_price)

    # Generate target column based on data type
    if data_type == "linear":
        df["SalePrice"] = noisy_score
    elif data_type == "binary":
        # Use median split for better balance
        threshold = np.median(noisy_score)
        df["WillSell"] = (noisy_score >= threshold).astype(int)
    elif data_type == "multiclass":
        # Use equal-sized bins for better class distribution
        sorted_scores = np.sort(noisy_score)
        n_per_class = len(sorted_scores) // num_classes
        
        # Create thresholds that ensure roughly equal class sizes
        thresholds = []
        for i in range(1, num_classes):
            idx = min(i * n_per_class, len(sorted_scores) - 1)
            thresholds.append(sorted_scores[idx])
        
        # Assign classes based on these thresholds
        class_labels = np.zeros(len(noisy_score), dtype=int)
        for i, threshold in enumerate(thresholds):
            class_labels[noisy_score >= threshold] = i + 1
            
        # Generate descriptive labels
        if num_classes <= 10:
            label_templates = {
                2: ["Low", "High"],
                3: ["Low", "Medium", "High"],
                4: ["Very Low", "Low", "High", "Very High"],
                5: ["Very Low", "Low", "Medium", "High", "Very High"],
                6: ["Very Low", "Low", "Mid-Low", "Mid", "Mid-High", "High"],
                7: ["Very Low", "Low", "Mid-Low", "Mid", "Mid-High", "High", "Very High"],
                8: ["Extremely Low", "Very Low", "Low", "Mid-Low", "Mid", "Mid-High", "High", "Very High"],
                9: ["Extremely Low", "Very Low", "Low", "Lower-Mid", "Mid", "Upper-Mid", "High", "Very High", "Extremely High"],
                10: ["Extremely Low", "Very Low", "Low", "Lower-Mid", "Mid-Low", "Mid", "Mid-High", "Upper-Mid", "High", "Very High"]
            }
            labels = label_templates.get(num_classes, [f"Category {i+1}" for i in range(num_classes)])
        else:
            labels = [f"Category {i+1}" for i in range(num_classes)]
        
        # Map numeric classes to descriptive labels
        df["PriceCategory"] = [labels[class_labels[i]] for i in range(len(class_labels))]
    else:
        raise ValueError(f"Invalid data_type: {data_type}. Must be 'linear', 'binary', or 'multiclass'")

    # Save chunk to file
    os.makedirs(output_path, exist_ok=True)
    chunk_file = os.path.join(output_path, f"chunk_{chunk_index}.csv")
    df.to_csv(chunk_file, index=False)
    
    if verbose:
        complexity_desc = f" (Complexity: {complexity_level})" if complexity_level > 0 else ""
        print(f"✅ Chunk {chunk_index} saved{complexity_desc}: {chunk_file}")
    
    return chunk_file


def generate_house_pricing_data(
    num_records=1000,
    price_noise_percent=0,
    feature_noise_percent=0,
    data_type="linear",
    num_classes=3,
    complexity_level=0,
    output_file="",
    chunk_size=100_000,
    num_workers=1,
    merge_chunks=True,
    delete_chunks=True,
    verbose=True
):
    """
    Generate synthetic house pricing data with configurable non-linear complexity.
    
    Parameters:
    -----------
    num_records : int
        Number of records to generate
    price_noise_percent : float
        Percentage of noise to add to prices (0-100)
    feature_noise_percent : float
        Percentage of noise to add to features (0-100)
    data_type : str
        Type of target variable: "linear", "binary", or "multiclass"
    num_classes : int
        Number of classes for multiclass classification
    complexity_level : int (0-50)
        Controls the non-linear complexity of the generated data
    output_file : str
        Output file path (auto-generated if empty)
    chunk_size : int
        Size of each processing chunk
    num_workers : int
        Number of parallel workers
    merge_chunks : bool
        Whether to merge chunks into single file
    delete_chunks : bool
        Whether to delete chunk files after merging
    verbose : bool
        Whether to print progress messages
    """
    
    # Validate inputs
    if not isinstance(complexity_level, int) or complexity_level < 0 or complexity_level > 50:
        raise ValueError("complexity_level must be an integer between 0 and 50")
    
    if data_type not in ["linear", "binary", "multiclass"]:
        raise ValueError("data_type must be 'linear', 'binary', or 'multiclass'")
    
    if num_classes < 2:
        raise ValueError("num_classes must be at least 2")
    
    # Set up random seed
    seed = int(time.time())
    np.random.seed(seed)
    random.seed(seed)

    # Generate output file name if not provided
    if not output_file:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        complexity_suffix = f"_c{complexity_level}" if complexity_level > 0 else ""
        output_file = f"house_data_{data_type}_{num_records}recs{complexity_suffix}_{timestamp}.csv"

    # Set up directories
    output_dir = os.path.dirname(output_file) if os.path.dirname(output_file) else "."
    chunk_dir = os.path.join(output_dir, f"chunks_{datetime.now().strftime('%Y%m%d_%H%M%S')}")

    # Calculate chunks
    num_chunks = (num_records + chunk_size - 1) // chunk_size
    
    # Prepare arguments for each chunk
    chunk_args = []
    for i in range(num_chunks):
        actual_chunk_size = min(chunk_size, num_records - i * chunk_size)
        args = (i, actual_chunk_size, seed, price_noise_percent / 100.0, 
                feature_noise_percent / 100.0, data_type, num_classes, 
                chunk_dir, verbose, complexity_level)
        chunk_args.append(args)

    if verbose:
        complexity_desc = f" with complexity level {complexity_level}" if complexity_level > 0 else " (linear)"
        print(f"🔧 Generating {num_records} records in {num_chunks} chunks{complexity_desc}...")

    # Generate chunks
    if num_workers == 1 or num_chunks <= 1:
        chunk_files = [generate_chunk_data_to_file(args) for args in chunk_args]
    else:
        with mp.Pool(num_workers) as pool:
            chunk_files = pool.map(generate_chunk_data_to_file, chunk_args)

    if verbose:
        print(f"✅ {len(chunk_files)} chunk files created in: {chunk_dir}")

    # Merge chunks if requested
    if merge_chunks:
        if verbose:
            print("🔄 Merging chunk files into a single CSV...")
        
        # Read and combine all chunks
        dataframes = []
        for chunk_file in sorted(chunk_files):
            try:
                df_chunk = pd.read_csv(chunk_file)
                dataframes.append(df_chunk)
            except Exception as e:
                print(f"⚠️ Error reading chunk {chunk_file}: {e}")
        
        if dataframes:
            combined_df = pd.concat(dataframes, ignore_index=True)
            
            # Ensure output directory exists
            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            
            # Save combined file
            combined_df.to_csv(output_file, index=False)
            
            if verbose:
                complexity_desc = f" (complexity level {complexity_level})" if complexity_level > 0 else ""
                print(f"✅ Merged file saved as{complexity_desc}: {output_file}")
                print(f"📊 Final dataset shape: {combined_df.shape}")

        # Clean up chunk files if requested
        if delete_chunks:
            if verbose:
                print("🗑️ Deleting chunk files...")
            for chunk_file in chunk_files:
                try:
                    os.remove(chunk_file)
                except Exception as e:
                    if verbose:
                        print(f"⚠️ Failed to delete {chunk_file}: {e}")
            
            # Try to remove chunk directory
            try:
                os.rmdir(chunk_dir)
                if verbose:
                    print(f"🧹 Deleted chunk directory: {chunk_dir}")
            except Exception as e:
                if verbose:
                    print(f"⚠️ Failed to delete chunk directory {chunk_dir}: {e}")
    else:
        if verbose:
            print("ℹ️ Skipping merge. Use `merge_chunks=True` to enable merging.")
    
    return output_file



In [52]:
def human_format(number, precision=1):
    """
    Convert a number into a human-readable format:
    1234 -> 1.2k
    1000000 -> 1m
    850 -> 850
    """
    suffixes = ['', 'k', 'm', 'b', 't']
    num = float(number)
    magnitude = 0

    while abs(num) >= 1000 and magnitude < len(suffixes) - 1:
        magnitude += 1
        num /= 1000.0

    if magnitude == 0:
        return str(int(number))  # show full number for values < 1000

    formatted = f"{num:.{precision}f}".rstrip('0').rstrip('.')
    return f"{formatted}{suffixes[magnitude]}"


In [53]:
t_noise = 0
f_noise = 0
records = 1000000
chunk_size = records // 1000
num_classes=10
complexity = 50



# Linear regression
filePath = "../data/NuralNetwork/linear/house_pricing_linear_"+str(human_format(records))+"_TN_"+str(t_noise)+"_FN_"+str(f_noise)+".csv"
generate_house_pricing_data(
    num_records=records,
    price_noise_percent=t_noise,
    feature_noise_percent=f_noise,
    output_file=filePath,
    chunk_size=chunk_size,
    complexity_level=complexity
)



# # Binary classification
# filePath = "../data/NuralNetwork/binary/house_pricing_binary_"+str(human_format(records))+"_TN_"+str(t_noise)+"_FN_"+str(f_noise)+".csv"
# generate_house_pricing_data(
#     num_records=records,
#     price_noise_percent=t_noise,
#     feature_noise_percent=f_noise,
#     output_file=filePath, 
#     data_type="binary",
#     chunk_size=chunk_size,
#     complexity_level=complexity)

# # Multi-class 
# filePath = "../data/NuralNetwork/multi-class/house_pricing_ multiclass_"+str(num_classes)+"_"+str(human_format(records))+"_TN_"+str(t_noise)+"_FN_"+str(f_noise)+".csv"
# generate_house_pricing_data(    
#     num_records=records,
#     price_noise_percent=t_noise,
#     feature_noise_percent=f_noise,
#     output_file=filePath, 
#     data_type="multiclass", 
#     num_classes=num_classes,
#     chunk_size=chunk_size,
#     complexity_level=complexity)

🔧 Generating 1000000 records in 1000 chunks with complexity level 50...
✅ Chunk 0 saved (Complexity: 50): ../data/NuralNetwork/linear\chunks_20250803_155220\chunk_0.csv
✅ Chunk 1 saved (Complexity: 50): ../data/NuralNetwork/linear\chunks_20250803_155220\chunk_1.csv
✅ Chunk 2 saved (Complexity: 50): ../data/NuralNetwork/linear\chunks_20250803_155220\chunk_2.csv
✅ Chunk 3 saved (Complexity: 50): ../data/NuralNetwork/linear\chunks_20250803_155220\chunk_3.csv
✅ Chunk 4 saved (Complexity: 50): ../data/NuralNetwork/linear\chunks_20250803_155220\chunk_4.csv
✅ Chunk 5 saved (Complexity: 50): ../data/NuralNetwork/linear\chunks_20250803_155220\chunk_5.csv
✅ Chunk 6 saved (Complexity: 50): ../data/NuralNetwork/linear\chunks_20250803_155220\chunk_6.csv
✅ Chunk 7 saved (Complexity: 50): ../data/NuralNetwork/linear\chunks_20250803_155220\chunk_7.csv
✅ Chunk 8 saved (Complexity: 50): ../data/NuralNetwork/linear\chunks_20250803_155220\chunk_8.csv
✅ Chunk 9 saved (Complexity: 50): ../data/NuralNetwork/

'../data/NuralNetwork/linear/house_pricing_linear_1m_TN_0_FN_0.csv'